# 10 — Setup Environment Skenario B (pip-only)

Target: **PC kampus** (JupyterHub, tanpa conda/venv). Jalankan SEKALI di kernel env Jupyter yang ada.

Semua install masuk ke env kernel saat ini — tidak bikin env terpisah (s2-main / s2-diffmot cukup satu).

Referensi: `docs/plans/2026-08-02-phase9-skenario-b-tracker.md`

In [ ]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

### 0. Cek python & GPU

`torch` yang sudah ada **tidak ditimpa** bila CUDA aktif (requirement DiffMOT tidak mengunci versi torch).
Kalau belum ada CUDA dan python ≤ 3.10, pakai sel fallback di bagian bawah.

In [ ]:
!nvidia-smi

In [ ]:
import sys
print("python:", sys.version.split()[0])
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU  :", torch.cuda.get_device_name(0))
except ImportError:
    print("torch belum terpasang — kalau env tidak punya CUDA dan python <= 3.10, jalankan sel fallback")

### 1. Paket Skenario B (deteksi, TrackEval, runner, plot)

In [ ]:
!pip install -q ultralytics trackeval motmetrics filterpy loguru pandas openpyxl matplotlib huggingface_hub

### 2. Dependency DiffMOT

`requirement.txt` DiffMOT berisi einops/lap/cython-bbox/fvcore dll — **tidak mengunci torch**.

Pitfall: `cython-bbox` rawan gagal build → ulangi dengan `pip install cython-bbox --no-build-isolation`.

In [ ]:
!pip install -q einops pyyaml easydict tensorboardX tqdm opencv-python scipy lap cython fvcore

In [ ]:
!pip install -q cython-bbox

### 3. Clone DiffMOT / OC_SORT / TrackEval (sekali)

In [ ]:
!git clone --depth 1 https://github.com/Kroery/DiffMOT $S2_EXT/diffmot
!git clone --depth 1 https://github.com/noahcao/OC_SORT $S2_EXT/OC_SORT
!git clone --depth 1 https://github.com/JonathonLuiten/TrackEval $S2_EXT/TrackEval
!cd $S2_EXT/diffmot && git submodule update --init --recursive

### 4. YOLOX / deep-person-reid / fast_reid (editable)

YOLOX **tidak dipakai untuk deteksi** (deteksi = YOLO26 fine-tune Skenario A lewat notebook 30; DiffMOT hanya membaca file deteksi), tapi kode DiffMOT mengimpornya saat load — wajib di-install.

In [ ]:
!cd $S2_EXT/diffmot/external/YOLOX && pip install -q -r requirements.txt && pip install -q -e .

In [ ]:
!cd $S2_EXT/diffmot/external/deep-person-reid && pip install -q -r requirements.txt && pip install -q -e .

In [ ]:
!cd $S2_EXT/diffmot/external/fast_reid && pip install -q -r docs/requirements.txt

### 5. Bobot DiffMOT (release v1.0)

- Motion D²MP: `MOT_epoch800.pt` → rename `mot_epoch800.pt`; `DanceTrack_epoch800.pt` → rename `dancetrack_epoch800.pt`
- ReID (FastReID): `mot20_sbs_S50.pth`, `dance_sbs_S50.pth`

Posisi checkpoint motion = `{diffmot}/experiments/{eval_expname}/{dataset}_epoch{epoch}.pt` (konvensi `diffmot.py::_build_model`).

In [ ]:
!mkdir -p $S2_EXT/diffmot/experiments/diffmot_mot $S2_EXT/diffmot/experiments/diffmot_dance $S2_EXT/diffmot/external/weights

In [ ]:
!wget -qO $S2_EXT/diffmot/experiments/diffmot_mot/mot_epoch800.pt https://github.com/Kroery/DiffMOT/releases/download/v1.0/MOT_epoch800.pt
!wget -qO $S2_EXT/diffmot/experiments/diffmot_dance/dancetrack_epoch800.pt https://github.com/Kroery/DiffMOT/releases/download/v1.0/DanceTrack_epoch800.pt
!wget -qO $S2_EXT/diffmot/external/weights/mot20_sbs_S50.pth https://github.com/Kroery/DiffMOT/releases/download/v1.0/mot20_sbs_S50.pth
!wget -qO $S2_EXT/diffmot/external/weights/dance_sbs_S50.pth https://github.com/Kroery/DiffMOT/releases/download/v1.0/dance_sbs_S50.pth

### 6. Verifikasi

In [ ]:
!ls -lh $S2_EXT/diffmot/experiments/diffmot_mot/ $S2_EXT/diffmot/experiments/diffmot_dance/ $S2_EXT/diffmot/external/weights/

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

### Fallback: pin torch 2.0.1 cu118

Hanya jalankan bila DiffMOT error saat run (notebook 50). Syarat python ≤ 3.10; wajib dari index cu118 (wheel PyPI 2.0.1 rusak: hilang dependensi nvidia → `libnvrtc not found`).

In [ ]:
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

**Lanjut**: `20_s2_download_data.ipynb` (data) → `30_s2_gen_detections.ipynb` (deteksi YOLO26).